# Pertemuan 10 — Algoritma Klasifikasi (Bagian 2)

Nama: Widya Anggara
NIM: 230401020091
Kelas: IF401


## Load Data CSV

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("./dataset/telco_churn.csv")

print("Shape dataset:", df.shape)
print("\n5 data pertama:")
display(df.head())

print("\nTipe data:")
print(df.dtypes)

print("\nProporsi Churn:")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(4))


Shape dataset: (7043, 21)

5 data pertama:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Tipe data:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Proporsi Churn:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64


## Preprocessing


In [10]:
from sklearn.model_selection import train_test_split

data = df.copy()

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

print("Missing TotalCharges sebelum penanganan:", data["TotalCharges"].isna().sum())

data["TotalCharges"] = data["TotalCharges"].fillna(data["TotalCharges"].median())

y = data["Churn"].map({"No": 0, "Yes": 1})

X = data.drop(columns=["Churn", "customerID"])

X = pd.get_dummies(X, dtype=int)

print("Jumlah fitur setelah encoding:", X.shape[1])
display(X.head())


Missing TotalCharges sebelum penanganan: 11
Jumlah fitur setelah encoding: 45


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,1,0,0,1,1,0,...,0,1,0,0,0,1,0,0,1,0
1,0,34,56.95,1889.50,0,1,1,0,1,0,...,0,0,1,0,1,0,0,0,0,1
2,0,2,53.85,108.15,0,1,1,0,1,0,...,0,1,0,0,0,1,0,0,0,1
3,0,45,42.30,1840.75,0,1,1,0,1,0,...,0,0,1,0,1,0,1,0,0,0
4,0,2,70.70,151.65,1,0,1,0,1,0,...,0,1,0,0,0,1,0,0,1,0


In [11]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Data train:", X_tr.shape)
print("Data test :", X_te.shape)

print("\nProporsi churn pada train:")
print(y_tr.value_counts(normalize=True).round(4))

print("\nProporsi churn pada test:")
print(y_te.value_counts(normalize=True).round(4))


Data train: (5634, 45)
Data test : (1409, 45)

Proporsi churn pada train:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64

Proporsi churn pada test:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64


## Latih Model Random Forest

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

print("Random Forest selesai dilatih.")


Random Forest selesai dilatih.


## Evaluasi Model



In [13]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Prediksi kelas dan probabilitas churn
y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

print("=== Classification Report ===")
print(classification_report(
    y_te,
    y_pred,
    target_names=["Tidak Churn", "Churn"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_te, y_pred))

print(f"Accuracy : {accuracy_score(y_te, y_pred):.3f}")
print(f"Precision: {precision_score(y_te, y_pred):.3f}")
print(f"Recall   : {recall_score(y_te, y_pred):.3f}")
print(f"F1-Score : {f1_score(y_te, y_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_te, y_proba):.3f}")
print(f"PR-AUC   : {average_precision_score(y_te, y_proba):.3f}")


=== Classification Report ===
              precision    recall  f1-score   support

 Tidak Churn       0.86      0.82      0.84      1035
       Churn       0.55      0.63      0.59       374

    accuracy                           0.77      1409
   macro avg       0.70      0.72      0.71      1409
weighted avg       0.78      0.77      0.77      1409

Confusion Matrix:
[[845 190]
 [140 234]]
Accuracy : 0.766
Precision: 0.552
Recall   : 0.626
F1-Score : 0.586
ROC-AUC  : 0.822
PR-AUC   : 0.604


### Feature Importance

In [14]:
feature_importance = pd.DataFrame({
    "Fitur": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance.head(10))


,Fitur,Importance
3,TotalCharges,0.142474
1,tenure,0.132329
2,MonthlyCharges,0.122295
36,Contract_Month-to-month,0.072302
18,OnlineSecurity_No,0.036642
38,Contract_Two year,0.036527
27,TechSupport_No,0.031064
43,PaymentMethod_Electronic check,0.029037
16,InternetService_Fiber optic,0.028522
21,OnlineBackup_No,0.016718


## Prediksi Probabilitas Churn

In [15]:
hasil_prediksi = pd.DataFrame({
    "customerID": df.loc[X_te.index, "customerID"],
    "Aktual_Churn": y_te.map({0: "No", 1: "Yes"}),
    "Prediksi_Churn": pd.Series(y_pred, index=X_te.index).map({0: "No", 1: "Yes"}),
    "Probabilitas_Churn": y_proba
}, index=X_te.index)

hasil_prediksi = hasil_prediksi.sort_values(
    "Probabilitas_Churn",
    ascending=False
)

print("10 pelanggan dengan probabilitas churn tertinggi:")
display(hasil_prediksi.head(10))


10 pelanggan dengan probabilitas churn tertinggi:


,customerID,Aktual_Churn,Prediksi_Churn,Probabilitas_Churn
1739,9804-ICWBG,Yes,Yes,1.000000
2927,5542-TBBWB,No,Yes,1.000000
1731,8375-DKEBR,Yes,Yes,1.000000
6623,9248-OJYKK,Yes,Yes,1.000000
1144,0841-NULXI,Yes,Yes,1.000000
2194,2514-GINMM,Yes,Yes,0.996667
809,1820-TQVEV,Yes,Yes,0.993333
6633,4415-IJZTP,Yes,Yes,0.993333
1081,1751-NCDLI,No,Yes,0.990000
6322,8752-STIVR,Yes,Yes,0.986667


## Kesimpulan

Dataset Telco Customer Churn bersifat **imbalanced**, karena pelanggan yang churn hanya sekitar 26,5% dari seluruh data. Random Forest dengan `class_weight="balanced"` memperoleh Accuracy sekitar **78,1%** dan ROC-AUC sekitar **82,2%**, tetapi Recall kelas churn pada threshold default masih sekitar **47,1%**.
